# Molecular Mechanics Core: topology → force terms → trajectory

This notebook is the compact MD entry point.  It uses a water-like toy molecule
to connect the theory to the package API:

$$
E_\mathrm{MM}
= E_\mathrm{bond}
+ E_\mathrm{angle}
+ E_\mathrm{dihedral}
+ E_\mathrm{LJ}
+ E_\mathrm{Coulomb}.
$$

The point is not chemical realism.  The point is to verify the data model:
typed atoms, topology, bonded terms, nonbonded exclusions, constraints, and
per-term energy diagnostics.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not locate repository root")


ROOT = find_repo_root()


## Reduced-unit Hamiltonian used here

The MD layer uses reduced internal units for these examples.  That means the
time, mass, length, charge, and energy scales are chosen by the model instead
of SI constants.  In this convention the simulated Hamiltonian is simply

$$
H(r,p) = K(p) + U(r),
\qquad
K(p) = \sum_i \frac{|p_i|^2}{2m_i}
      = \frac{1}{2}\sum_i m_i |v_i|^2.
$$

The force on atom \(i\) is the negative gradient of the potential energy:

$$
F_i = -\nabla_i U(r).
$$

For a correct NVE integrator, \(H\) should remain nearly constant.  Small
oscillatory drift is expected from finite timestep integration; systematic
drift usually means the timestep is too large, constraints are loose, or the
force implementation is wrong.


## Build a typed system

The topology says which atoms are bonded and which angle is meaningful.  The
force field maps atom types to parameters.  Bonded pairs are excluded from the
nonbonded pair list, while the H–H pair remains active.


### Force-field terms in this toy system

The water-like system below uses two O–H harmonic bonds and one H–O–H harmonic
angle:

$$
E_\mathrm{bond}
= \frac{1}{2}\sum_{(ij)}
k_{ij}\left(r_{ij}-r_{ij}^{0}\right)^2,
\qquad
r_{ij}=|r_i-r_j|.
$$

$$
E_\mathrm{angle}
= \frac{1}{2}\sum_{(ijk)}
k_{ijk}\left(\theta_{ijk}-\theta_{ijk}^{0}\right)^2.
$$

The nonbonded term is evaluated only for non-excluded pairs.  For a pair
\((ij)\),

$$
E_{ij}^\mathrm{LJ}
=4\epsilon_{ij}\left[
\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12}
-\left(\frac{\sigma_{ij}}{r_{ij}}\right)^6
\right],
\qquad
E_{ij}^\mathrm{Coulomb}
= k_e\frac{q_iq_j}{r_{ij}}.
$$

In a real force field these parameters would come from AMBER, CHARMM, OPLS,
GROMOS, or another parameter source.  Here they are programmatic so the data
model remains inspectable.


In [ ]:
from mlx_atomistic import (
    AngleParameter,
    AtomType,
    BondParameter,
    Cell,
    DistanceConstraints,
    ForceField,
    MMSystem,
    NonbondedParameter,
    Topology,
)
from mlx_atomistic.md import SimulationConfig, simulate_nve

cell = Cell.orthorhombic((8.0, 8.0, 8.0))
topology = Topology.from_sequences(
    n_atoms=3,
    bonds=[(0, 1), (0, 2)],
    angles=[(1, 0, 2)],
    partial_charges=[-0.8, 0.4, 0.4],
)

positions = np.array(
    [
        [4.000, 4.000, 4.000],
        [4.957, 4.000, 4.000],
        [3.760, 4.927, 4.000],
    ],
    dtype=np.float32,
)
velocities = np.array(
    [
        [0.000, 0.010, 0.000],
        [0.000, -0.020, 0.000],
        [0.000, 0.010, 0.000],
    ],
    dtype=np.float32,
)

force_field = ForceField(
    atom_types=[AtomType("O", 16.0), AtomType("H", 1.0)],
    nonbonded=[
        NonbondedParameter("O", sigma=1.0, epsilon=0.20),
        NonbondedParameter("H", sigma=0.6, epsilon=0.04),
    ],
    bonds=[BondParameter(("O", "H"), k=250.0, length=0.957)],
    angles=[AngleParameter(("H", "O", "H"), k=40.0, angle=np.deg2rad(104.5))],
    cutoff=3.0,
    lj_shift=True,
)
system = MMSystem.from_sequences(
    symbols=["O", "H", "H"],
    atom_types=["O", "H", "H"],
    positions=positions,
    velocities=velocities,
    topology=topology,
    masses=force_field.masses_for(["O", "H", "H"]),
    charges=[-0.8, 0.4, 0.4],
    cell=cell,
)
terms = force_field.build_force_terms(system)

print("force terms:", [getattr(term, "name", type(term).__name__) for term in terms])
print("topology exclusions:", np.asarray(topology.exclusions).tolist())
print("nonbonded pairs:", np.asarray(topology.nonbonded_pairs()).tolist())


## Run constrained NVE

Velocity Verlet updates the microcanonical trajectory.  SHAKE/RATTLE-style
pair-distance constraints keep the two O–H distances close to the target:

$$
H = K(p) + U(r), \qquad \Delta E(t)=E(t)-E(0).
$$


### What SHAKE/RATTLE is enforcing

Each constrained pair defines a holonomic constraint

$$
g_a(r) = |r_i-r_j|^2 - d_{ij}^2 = 0.
$$

SHAKE corrects positions after the unconstrained position drift so that
\(g_a(r)\approx 0\).  RATTLE corrects velocities so the velocity is tangent to
the constraint surface:

$$
\dot g_a(r,v)
= 2(r_i-r_j)\cdot(v_i-v_j)
\approx 0.
$$

The plotted `constraint_max_error` is the largest residual distance error after
the correction step.  It should be close to the configured tolerance and should
not grow over time.


In [ ]:
constraints = DistanceConstraints(
    pairs=[(0, 1), (0, 2)],
    distances=[0.957, 0.957],
    tolerance=1e-5,
    max_iterations=40,
)
result = simulate_nve(
    system.positions,
    system.velocities,
    masses=system.masses,
    cell=cell,
    force_terms=terms,
    config=SimulationConfig(dt=0.0005, steps=300, sample_interval=10),
    constraints=constraints,
)

mx.eval(result.total_energy, result.constraint_max_error)
times = np.arange(result.total_energy.shape[0]) * 0.0005
energy = np.asarray(result.total_energy)
drift = energy - energy[0]
constraint_error = np.asarray(result.constraint_max_error)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(times, drift)
axes[0].set_title("NVE total-energy drift")
axes[0].set_xlabel("time / reduced units")
axes[0].set_ylabel("E(t) - E(0)")
axes[1].plot(times, constraint_error)
axes[1].set_title("maximum constraint error")
axes[1].set_xlabel("time / reduced units")
axes[1].set_ylabel("|r_ij - d_ij|")
fig.tight_layout()


## Inspect energy decomposition and trajectory geometry

The dense scalar diagnostics preserve every integration step, while positions
are sampled sparsely.  That is the intended production pattern: keep scalar
diagnostics cheap and dense, but avoid storing every coordinate frame unless the
workflow needs it.


### Reading the plots

The left plot should answer: which physical term is dominating the potential
energy?  The right plot should answer: are the imposed distances actually held
fixed in the sampled trajectory?

For this toy system the H–H pair is not bonded, so it remains in the nonbonded
path.  The O–H bonded pairs are removed from ordinary LJ/Coulomb evaluation to
avoid double-counting short-range interactions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, series in result.potential_energy_by_term.items():
    axes[0].plot(times, np.asarray(series), label=name)
axes[0].set_title("potential-energy components")
axes[0].set_xlabel("time / reduced units")
axes[0].set_ylabel("energy")
axes[0].legend()

sampled = np.asarray(result.sampled_positions)
bond_01 = np.linalg.norm(sampled[:, 0] - sampled[:, 1], axis=1)
bond_02 = np.linalg.norm(sampled[:, 0] - sampled[:, 2], axis=1)
axes[1].plot(np.asarray(result.sampled_time), bond_01, label="O-H 1")
axes[1].plot(np.asarray(result.sampled_time), bond_02, label="O-H 2")
axes[1].axhline(0.957, color="black", linewidth=1, linestyle="--")
axes[1].set_title("sampled constrained bond lengths")
axes[1].set_xlabel("time / reduced units")
axes[1].set_ylabel("distance")
axes[1].legend()
fig.tight_layout()


In [ ]:
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
frame0 = np.asarray(result.sampled_positions[0])
framef = np.asarray(result.sampled_positions[-1])
for frame, alpha, label in [(frame0, 0.35, "initial"), (framef, 1.0, "final")]:
    ax.scatter(frame[:, 0], frame[:, 1], frame[:, 2], s=[90, 45, 45], alpha=alpha, label=label)
    for i, j in [(0, 1), (0, 2)]:
        ax.plot(
            [frame[i, 0], frame[j, 0]],
            [frame[i, 1], frame[j, 1]],
            [frame[i, 2], frame[j, 2]],
            color="tab:gray",
            alpha=alpha,
        )
ax.set_title("water-like constrained molecule")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.legend()
